# Лабораторна робота 5. Задача дискретного логарифма i протокол Дiффi-Хеллмана

## Л5.1 Реалiзуйте алгоритм знаходження твiрного елемента $a$ мультиплiкативної групи поля $GF(p)$, $p$ — непарне просте, бiнарна довжина якого не бiльша за 20.

Нехай $p$ — непарне просте число. Розглядаємо мультиплікативну групу поля

$$
GF(p) = \mathbb{Z}_p^* = \{1,2,\dots,p-1\},
$$

яка є циклічною групою порядку

$$
n = p - 1.
$$

Елемент $a \in GF(p)$ називається **твірним (генератором)**, якщо він породжує всю групу:

$$
G = \langle a \rangle.
$$

Це означає, що будь-який ненульовий елемент можна подати у вигляді:

$$
x \equiv a^s \pmod{p},
$$

для деякого цілого $s$.

---

# Ідея алгоритму

Нехай факторизація порядку групи має вигляд:

$$
n = p_1^{e_1} p_2^{e_2} \cdots p_k^{e_k}.
$$

Ключовий факт:

> Елемент $a$ є генератором тоді і тільки тоді, коли для всіх простих дільників $p_i$ виконується
>
> $$ a^{n/p_i} \not\equiv 1 \pmod{p}.$$

---

# Алгоритм знаходження генератора

Мета — знайти елемент $a$, для якого:

$$
\mathrm{ord}(a) = n.
$$

## Кроки:

1. Випадково вибираємо $a \in GF(p)$.

2. Для кожного простого дільника $p_i$ перевіряємо:

$$
b = a^{n/p_i} \pmod{p}.
$$

3. Якщо існує $p_i$ таке, що:

$$
b = 1,
$$

то $a$ не є генератором.

4. Якщо для всіх $p_i$:

$$
a^{n/p_i} \ne 1,
$$

то $a$ є твірним елементом.

In [1]:
import random
import math

def get_generator(p):
    # Порядок мультиплікативної групи GF(p) дорівнює n = p - 1
    n = p - 1

    # Знаходимо прості множники n = p_1^e_1*p_2^e_2*p_3^e_3*...p_k^e_k      
    prime_p_i = []
    temp_n = n
    divisor = 2
    
    # Продовжуємо, поки дільник не перевищить корінь з числа
    limit = math.isqrt(temp_n)
    while divisor <= limit:
        if temp_n % divisor == 0:
            prime_p_i.append(divisor) 
            # Видаляємо всі повторення цього множника
            while temp_n % divisor == 0:
                temp_n = temp_n // divisor
        divisor += 1
    
    if temp_n > 1:
        prime_p_i.append(temp_n)
    
    # Крок 1: Вибираємо випадковий елемент a
    while True:
        a = random.randint(2, p - 1)
        is_generator = True
        
        # Крок 2: Перевірка умови a^(n/p_i) != 1 mod p
        for pi in prime_p_i:
            # Обчислюємо b = a^(n/pi) mod p
            if pow(a, n // pi, p) == 1:
                is_generator = False
                break
        
        # Крок 3: Якщо умова виконана для всіх множників
        if is_generator:
            return a


p = 433
gen = get_generator(p)
print(f"Для p = {p}, знайдений твірний елемент a = {gen}")

Для p = 433, знайдений твірний елемент a = 319


## Л5.2 Реалiзуйте алгоритм малих i великих крокiв розв’язування задачi дискретного логарифма в групi $G = ⟨a⟩$.

Алгоритм **Baby-Step-Giant-Step (BSGS)** — це детермінований алгоритм для розв’язування задачі дискретного логарифма в циклічній групі. 

## Постановка задачі

Нехай

$$
G = \langle a \rangle
$$

— циклічна група порядку $n$, а $a$ — її твірний елемент.

Тоді будь-який елемент $x \in G$ можна подати у вигляді

$$
x = a^s,
$$

де $s$ — деяке ціле число.

Число $s$ називається **дискретним логарифмом** елемента $x$ за основою $a$:

$$
s = \log_a(x) \pmod{n}.
$$

Задача дискретного логарифма полягає у знаходженні числа $s$, якщо відомі $a$ та $x$.

---

## Ідея алгоритму

Алгоритм BSGS базується на представленні показника степеня у вигляді

$$
s = j + mi,
$$

де

$$
m \approx \sqrt{n},
$$

а

$$
0 \le i,j \le m.
$$

Тоді

$$
x = a^s = a^{j+mi}.
$$

Перепишемо це рівняння:

$$
a^j = x \cdot a^{-mi}.
$$

Алгоритм будує дві таблиці:

- **Baby Steps** — значення
  $$
  a^j \pmod{p}
  $$
- **Giant Steps** — значення
  $$
  x \cdot a^{-mi} \pmod{p}
  $$

Коли знаходиться збіг між двома таблицями, отримуємо

$$
a^j = x \cdot a^{-mi},
$$

а отже

$$
x = a^{j+mi}.
$$

Тому

$$
\log_a(x) = j + mi.
$$

---

# Приклад

Нехай

$$
p = 433,
\qquad
a = 7,
\qquad
x = 166.
$$

Потрібно знайти

$$
\log_7(166).
$$

Оскільки

$$
m \approx \sqrt{433} \approx 21,
$$

беремо

$$
m = 21.
$$

---

## Baby Steps

Обчислюємо

$$
a^j \pmod{433},
\qquad
0 \le j \le 21.
$$

| $j$ | $7^j \bmod 433$ |
|---|---|
| 0 | 1 |
| 1 | 7 |
| 2 | 49 |
| 3 | 343 |
| 4 | 236 |
| 5 | 353 |
| 6 | 306 |
| 7 | 410 |
| 8 | 272 |
| 9 | 172 |
| 10 | 338 |
| 11 | 201 |
| 12 | 108 |
| 13 | 323 |
| 14 | 96 |
| 15 | 239 |
| 16 | 374 |
| 17 | 20 |
| 18 | 140 |
| 19 | 114 |
| 20 | 365 |
| 21 | 292 |

---

## Giant Steps

Обчислюємо

$$
x \cdot a^{-mi} \pmod{433}.
$$

Під час обчислень отримуємо:

$$
166,
$$

$$
166 \cdot 7^{-21},
$$

$$
166 \cdot 7^{-21\cdot 2} \equiv 353 \pmod{433}.
$$

Помічаємо, що

$$
353
$$

вже є у таблиці Baby Steps:

$$
7^5 \equiv 353 \pmod{433}.
$$

Отже,

$$
j = 5,
\qquad
i = 2.
$$

Тоді

$$
\log_7(166)
=
5 + 21 \cdot 2
=
47.
$$

Перевірка:

$$
7^{47} \equiv 166 \pmod{433}.
$$


In [2]:
def baby_step_giant_step(a, x, p):
    n = p - 1
    # +1 - округлення до найближчого більшого цілого
    m = math.isqrt(n) + 1

    baby_steps = {}
    for j in range(m):
        value = pow(a, j, p)
        baby_steps[value] = j

    # a^(-m) mod p, спочатку обчислюємо a^m mod p, потім обернений до нього
    a_inv_m = pow(pow(a, m, p), -1, p)

    gamma = x
    for i in range(m):
        # Перевіряємо збіг
        if gamma in baby_steps:
            j = baby_steps[gamma]
            s = j + i * m
            return s
        # Наступний giant step, % - остача від ділення
        gamma = (gamma * a_inv_m) % p
    return None


p = 433
a = 7
x = 166
result = baby_step_giant_step(a, x, p)
print(f"log_{a}({x}) mod {p} = {result}")
print(f"Перевірка: {a}^{result} = {pow(a, result, p)} mod {p}")

log_7(166) mod 433 = 47
Перевірка: 7^47 = 166 mod 433


## Л5.3 Реалiзуйте протокол Дiффi-Хеллмана на основi групи $G$.

Протокол **Diffie–Hellman Key Exchange** — це криптографічний протокол узгодження спільного секретного ключа через відкритий канал зв’язку.

Протокол дозволяє двом сторонам:

- Алісі $(A)$
- Бобу $(B)$

створити спільний секретний ключ, навіть якщо всі повідомлення передаються відкритим каналом.

---

# Математична основа

Нехай

$$
G = \langle a \rangle
$$

— циклічна група порядку $n$, а

$$
a
$$

— твірний елемент групи.

У випадку мультиплікативної групи поля

$$
GF(p)
$$

порядок $n = p - 1$.

---

# Одноразове налаштування

Учасники публічно домовляються про:

- велике просте число

$$
p,
$$

- твірний елемент

$$
a,
\qquad
2 \le a \le p-2.
$$

Ці значення не є секретними.

---

# Етапи протоколу

## 1. Вибір секретів

### Аліса

Обирає випадкове секретне число

$$
x,
\qquad
1 \le x \le p-2.
$$

### Боб

Обирає випадкове секретне число

$$
y,
\qquad
1 \le y \le p-2.
$$

---

# 2. Обмін повідомленнями

### Аліса обчислює

$$
A = a^x \pmod{p}
$$

та надсилає Бобу.

---

### Боб обчислює

$$
B = a^y \pmod{p}
$$

та надсилає Алісі.

---

# 3. Обчислення спільного ключа

## Аліса

Після отримання $B$ обчислює:

$$
K = B^x \pmod{p}.
$$

---

## Боб

Після отримання $A$ обчислює:

$$
K = A^y \pmod{p}.
$$

---

# Чому ключі однакові

Аліса отримує:

$$
K = (a^y)^x \pmod{p}
$$

тобто

$$
K = a^{yx} \pmod{p}.
$$

---

Боб отримує:

$$
K = (a^x)^y \pmod{p}
$$

тобто

$$
K = a^{xy} \pmod{p}.
$$

---

Оскільки

$$
xy = yx,
$$

обидві сторони отримують однаковий секретний ключ:

$$
K = a^{xy} \pmod{p}.
$$

---

# Безпека протоколу

Зловмисник бачить лише:

$$
p,
\quad
a,
\quad
a^x,
\quad
a^y.
$$

Щоб знайти спільний ключ, потрібно визначити:

$$
x
\quad \text{або} \quad y,
$$

тобто розв’язати задачу дискретного логарифма:

$$
a^x \equiv A \pmod{p}.
$$

Для великих чисел ця задача є обчислювально складною.

---

# Приклад

Нехай:

$$
p = 23,
\qquad
a = 5.
$$

---

## Крок 1. Секрети

Аліса обирає:

$$
x = 6.
$$

Боб обирає:

$$
y = 15.
$$

---

## Крок 2. Відкриті повідомлення

### Аліса:

$$
A = 5^6 \bmod 23 = 8.
$$

### Боб:

$$
B = 5^{15} \bmod 23 = 19.
$$

---

## Крок 3. Обчислення ключа

### Аліса:

$$
K = 19^6 \bmod 23 = 2.
$$

### Боб:

$$
K = 8^{15} \bmod 23 = 2.
$$

---

Обидві сторони отримали однаковий секретний ключ:

$$
K = 2.
$$

In [3]:
def diffie_hellman(p, x=None, y=None, verbose=True):

    a = get_generator(p)

    if a is None:
        raise ValueError("Твірний не знайдено для цього p")

    # секрети
    if x is None:
        x = random.randint(1, p - 2)
    if y is None:
        y = random.randint(1, p - 2)

    # відкриті значення
    A = pow(a, x, p)
    B = pow(a, y, p)

    # спільний ключ
    K_alice = pow(B, x, p)
    K_bob = pow(A, y, p)

    if verbose:
        print("p =", p)
        print("generator a =", a)
        print("secret x =", x)
        print("secret y =", y)

        print("\nA =", A)
        print("B =", B)

        print("\nK alice =", K_alice)
        print("K bob   =", K_bob)

        print("\nОднаковий секретний ключ?:", K_alice == K_bob)

    return {
        "p": p,
        "a": a,
        "x": x,
        "y": y,
        "A": A,
        "B": B,
        "shared_key": K_alice
    }

result = diffie_hellman(p=23)

p = 23
generator a = 15
secret x = 10
secret y = 21

A = 3
B = 20

K alice = 8
K bob   = 8

Однаковий секретний ключ?: True
